# Sales Forecast Training (Prophet)

Trains company- and material-level Prophet models using gold feature tables and writes results to **`jm_databricks_learning_ws.ml`**.

**Inputs (gold / serve)**
- `serve.company_forecast_features_monthly`
- `serve.forecast_features_monthly`

**Outputs (ml)**
- `ml.forecast_runs`
- `ml.forecast_backtest`
- `ml.forecast_backtest_detail`
- `ml.sales_forecast_monthly`

**Prerequisites**
1. Silver + gold pipelines have run successfully
2. `ml` schema is created by the setup cell in this notebook (no manual DDL required)
3. Run **cell 1 first** — it installs `prophet` + `mlflow` and restarts Python

**Tip:** Databricks Runtime ML includes MLflow pre-installed; standard serverless may need the pip cell.

In [ ]:
%pip install prophet mlflow
dbutils.library.restartPython()

## Configuration

In [ ]:
import json
from datetime import datetime, timezone

import mlflow
import numpy as np
import pandas as pd
from prophet import Prophet

CATALOG = "jm_databricks_learning_ws"
SERVE_SCHEMA = "serve"
ML_SCHEMA = "ml"

SERVE = f"{CATALOG}.{SERVE_SCHEMA}"
ML = f"{CATALOG}.{ML_SCHEMA}"

TARGET_COLUMN = "sales_quantity"
TEST_MONTHS = 6
FORECAST_HORIZON_MONTHS = 12
TOP_MATERIALS_LIMIT = 10

REGRESSORS = (
    "purchase_quantity",
    "production_output_quantity",
    "inventory_on_hand",
    "downtime_hours",
)

PROPHET_PARAMS_COMPANY = {
    "seasonality_mode": "multiplicative",
    "yearly_seasonality": True,
    "weekly_seasonality": False,
    "daily_seasonality": False,
    "changepoint_prior_scale": 0.1,
    "seasonality_prior_scale": 10.0,
}

PROPHET_PARAMS_MATERIAL = {
    **PROPHET_PARAMS_COMPANY,
    "seasonality_mode": "additive",  # more stable for intermittent SKU demand
}

MLFLOW_EXPERIMENT = "/Users/yogirajsinh.parmar@jayseaflux.onmicrosoft.com/Sales_Forecast_Prophet_Regressors"
MODEL_NAME = "prophet_with_regressors"

print(f"Serve tables : {SERVE}.*")
print(f"ML tables    : {ML}.*")

## Create ML tables (idempotent)

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ML}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ML}.forecast_runs (
    run_id BIGINT,
    run_name STRING,
    as_of_date DATE,
    grain STRING,
    target_column STRING,
    model_name STRING,
    horizon_months INT,
    test_months INT,
    top_materials_limit INT,
    params_json STRING,
    created_at TIMESTAMP
) USING DELTA
""")

for table, ddl in {
    "forecast_backtest": """
        run_id BIGINT, grain_id STRING, metric_name STRING, metric_value DOUBLE
    """,
    "forecast_backtest_detail": """
        run_id BIGINT, grain_id STRING, forecast_month DATE,
        actual DOUBLE, predicted DOUBLE, error DOUBLE
    """,
    "sales_forecast_monthly": """
        run_id BIGINT, as_of_date DATE, grain STRING, grain_id STRING,
        forecast_month DATE, year_month STRING,
        yhat DOUBLE, yhat_lower DOUBLE, yhat_upper DOUBLE,
        model_name STRING, created_at TIMESTAMP
    """,
}.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {ML}.{table} ({ddl}) USING DELTA")

print("ML tables ready")

## Training helpers

In [ ]:
def next_run_id() -> int:
    row = spark.sql(f"SELECT COALESCE(MAX(run_id), 0) + 1 AS next_id FROM {ML}.forecast_runs").collect()
    return int(row[0]["next_id"])


def _metrics(actuals: np.ndarray, preds: np.ndarray) -> dict[str, float]:
    actuals = np.asarray(actuals, dtype=float)
    preds = np.asarray(preds, dtype=float)
    err = actuals - preds
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    nonzero = np.abs(actuals) > 1e-9
    if nonzero.any():
        mape = float(np.mean(np.abs(err[nonzero] / actuals[nonzero])) * 100)
        wmape = float(np.sum(np.abs(err[nonzero])) / np.sum(np.abs(actuals[nonzero])) * 100)
    else:
        mape = float("nan")
        wmape = float("nan")
    return {"mae": mae, "rmse": rmse, "mape": mape, "wmape": wmape}


def _fill_future_regressors(history: pd.DataFrame, future: pd.DataFrame, regressors: list[str]) -> pd.DataFrame:
    out = future.copy()
    hist = history.sort_values("ds").copy()
    for col in regressors:
        if col not in hist.columns:
            out[col] = 0.0
            continue
        hist_map = hist.set_index("ds")[col]
        values = []
        for ds in out["ds"]:
            if ds in hist_map.index:
                values.append(float(hist_map.loc[ds]))
                continue
            lag_year = ds - pd.DateOffset(years=1)
            if lag_year in hist_map.index:
                values.append(float(hist_map.loc[lag_year]))
                continue
            trailing = hist_map[hist_map.index < ds].tail(3)
            values.append(float(trailing.mean()) if len(trailing) else 0.0)
        out[col] = values
    return out


def _active_regressors(train_df: pd.DataFrame, regressors: tuple[str, ...]) -> list[str]:
    active = []
    for col in regressors:
        if col not in train_df.columns:
            continue
        series = train_df[col].astype(float)
        if series.nunique(dropna=True) <= 1:
            continue
        if float(series.std(skipna=True) or 0.0) <= 1e-9:
            continue
        active.append(col)
    return active


def _prepare_series(df: pd.DataFrame, target: str, regressors: tuple[str, ...]) -> pd.DataFrame:
    frame = df.copy()
    frame["ds"] = pd.to_datetime(frame["month_start_date"])
    frame["y"] = frame[target].astype(float).clip(lower=0.0)
    for col in regressors:
        if col not in frame.columns:
            frame[col] = 0.0
        frame[col] = frame[col].astype(float).fillna(0.0)
    return frame.sort_values("ds").reset_index(drop=True)


def _fit_prophet(train_df: pd.DataFrame, params: dict, regressors: tuple[str, ...]):
    active = _active_regressors(train_df, regressors)
    model = Prophet(**params)
    for col in active:
        model.add_regressor(col)
    model.fit(train_df[["ds", "y", *active]])
    return model, active


def _predict_horizon(model, history: pd.DataFrame, periods: int, regressors: list[str]) -> pd.DataFrame:
    future = model.make_future_dataframe(periods=periods, freq="MS")
    if regressors:
        future = _fill_future_regressors(history, future, regressors)
    forecast = model.predict(future)
    for col in ("yhat", "yhat_lower", "yhat_upper"):
        forecast[col] = forecast[col].clip(lower=0.0)
    return forecast


def _train_one_series(
    series: pd.DataFrame,
    *,
    grain_id: str,
    prophet_params: dict,
    test_months: int = TEST_MONTHS,
    horizon_months: int = FORECAST_HORIZON_MONTHS,
) -> tuple[dict[str, float], pd.DataFrame, pd.DataFrame, pd.Timestamp]:
    prepared = _prepare_series(series, TARGET_COLUMN, REGRESSORS)
    if len(prepared) <= test_months + 12:
        raise ValueError(f"{grain_id}: need > {test_months + 12} months, got {len(prepared)}")

    train = prepared.iloc[:-test_months].copy()
    test = prepared.iloc[-test_months:].copy()

    model_cv, active_cv = _fit_prophet(train, prophet_params, REGRESSORS)
    future_cv = _predict_horizon(model_cv, train, test_months, active_cv)
    preds = future_cv.set_index("ds")["yhat"].reindex(test["ds"].values).clip(lower=0.0)
    actuals = test.set_index("ds")["y"]
    metrics = _metrics(actuals.values, preds.values)

    detail = pd.DataFrame({
        "forecast_month": test["ds"].dt.date,
        "actual": actuals.values,
        "predicted": preds.values,
        "error": actuals.values - preds.values,
    })

    model_full, active_full = _fit_prophet(prepared, prophet_params, REGRESSORS)
    forecast = _predict_horizon(model_full, prepared, horizon_months, active_full)
    cutoff = prepared["ds"].max()
    return metrics, detail, forecast, cutoff


def _write_delta_table(table: str, pdf: pd.DataFrame) -> None:
    if pdf.empty:
        return
    spark.createDataFrame(pdf).write.mode("append").saveAsTable(f"{ML}.{table}")


def _insert_run(run_id: int, run_name: str, as_of_date, grain: str, params: dict) -> None:
    row = pd.DataFrame([{
        "run_id": run_id,
        "run_name": run_name,
        "as_of_date": pd.to_datetime(as_of_date).date(),
        "grain": grain,
        "target_column": TARGET_COLUMN,
        "model_name": MODEL_NAME,
        "horizon_months": FORECAST_HORIZON_MONTHS,
        "test_months": TEST_MONTHS,
        "top_materials_limit": TOP_MATERIALS_LIMIT if grain == "material" else None,
        "params_json": json.dumps({**params, "regressors": list(REGRESSORS)}),
        "created_at": datetime.now(timezone.utc),
    }])
    _write_delta_table("forecast_runs", row)


def _write_backtest(run_id: int, grain_id: str, metrics: dict[str, float], detail: pd.DataFrame) -> None:
    metric_rows = pd.DataFrame([
        {"run_id": run_id, "grain_id": grain_id, "metric_name": k, "metric_value": v}
        for k, v in metrics.items() if v == v
    ])
    _write_delta_table("forecast_backtest", metric_rows)

    detail_out = detail.copy()
    detail_out.insert(0, "run_id", run_id)
    detail_out.insert(1, "grain_id", grain_id)
    _write_delta_table("forecast_backtest_detail", detail_out)


def _write_forecast(
    run_id: int,
    as_of_date,
    grain: str,
    grain_id: str,
    forecast: pd.DataFrame,
    cutoff: pd.Timestamp,
) -> int:
    next_rows = forecast[forecast["ds"] > cutoff][["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
    if next_rows.empty:
        return 0
    out = pd.DataFrame({
        "run_id": run_id,
        "as_of_date": pd.to_datetime(as_of_date).date(),
        "grain": grain,
        "grain_id": grain_id,
        "forecast_month": next_rows["ds"].dt.date,
        "year_month": next_rows["ds"].dt.strftime("%Y-%m"),
        "yhat": next_rows["yhat"].astype(float),
        "yhat_lower": next_rows["yhat_lower"].astype(float),
        "yhat_upper": next_rows["yhat_upper"].astype(float),
        "model_name": MODEL_NAME,
        "created_at": datetime.now(timezone.utc),
    })
    _write_delta_table("sales_forecast_monthly", out)
    return len(out)

## Load gold feature tables

In [ ]:
company_df = (
    spark.table(f"{SERVE}.company_forecast_features_monthly")
    .orderBy("month_start_date")
    .toPandas()
)
material_df = (
    spark.table(f"{SERVE}.forecast_features_monthly")
    .orderBy("material_id", "month_start_date")
    .toPandas()
)

if company_df.empty:
    raise ValueError(f"{SERVE}.company_forecast_features_monthly is empty. Re-run gold pipeline.")
if material_df.empty:
    raise ValueError(f"{SERVE}.forecast_features_monthly is empty. Re-run gold pipeline.")

company_df["month_start_date"] = pd.to_datetime(company_df["month_start_date"])
material_df["month_start_date"] = pd.to_datetime(material_df["month_start_date"])

as_of_date = company_df["month_start_date"].max().date()
print(f"Company rows : {len(company_df)}")
print(f"Material rows: {len(material_df)}  |  SKUs: {material_df['material_id'].nunique()}")
print(f"As-of date   : {as_of_date}")
display(company_df.tail(3))

## Train company-level model

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)

company_run_id = next_run_id()
company_run_name = f"company_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

with mlflow.start_run(run_name=company_run_name) as run:
    mlflow.log_params(PROPHET_PARAMS_COMPANY)
    mlflow.log_params({
        "grain": "company",
        "target": TARGET_COLUMN,
        "test_months": TEST_MONTHS,
        "horizon_months": FORECAST_HORIZON_MONTHS,
        "regressors": ",".join(REGRESSORS),
    })

    company_metrics, company_detail, company_forecast, company_cutoff = _train_one_series(
        company_df,
        grain_id="COMPANY",
        prophet_params=PROPHET_PARAMS_COMPANY,
    )

    _insert_run(company_run_id, company_run_name, as_of_date, "company", PROPHET_PARAMS_COMPANY)
    _write_backtest(company_run_id, "COMPANY", company_metrics, company_detail)
    company_forecast_rows = _write_forecast(
        company_run_id, as_of_date, "company", "COMPANY", company_forecast, company_cutoff
    )

    for metric_name, metric_value in company_metrics.items():
        if metric_value == metric_value:
            mlflow.log_metric(f"test_{metric_name}", metric_value)
    mlflow.log_metric("forecast_rows", company_forecast_rows)
    mlflow.set_tag("run_id", str(company_run_id))

print(f"Company run_id={company_run_id}")
print(
    f"  MAE={company_metrics['mae']:.0f}  RMSE={company_metrics['rmse']:.0f}  "
    f"MAPE={company_metrics['mape']:.1f}%  wMAPE={company_metrics['wmape']:.1f}%"
)
print(f"  Forecast rows written: {company_forecast_rows}")

display(company_detail.round(1))

## Train material-level models (top SKUs)

In [ ]:
material_run_id = next_run_id()
material_run_name = f"material_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
material_results = []

with mlflow.start_run(run_name=material_run_name) as run:
    mlflow.log_params(PROPHET_PARAMS_MATERIAL)
    mlflow.log_params({
        "grain": "material",
        "target": TARGET_COLUMN,
        "test_months": TEST_MONTHS,
        "horizon_months": FORECAST_HORIZON_MONTHS,
        "top_materials_limit": TOP_MATERIALS_LIMIT,
        "regressors": ",".join(REGRESSORS),
    })

    _insert_run(material_run_id, material_run_name, as_of_date, "material", PROPHET_PARAMS_MATERIAL)

    for material_id, group in material_df.groupby("material_id"):
        try:
            metrics, detail, forecast, cutoff = _train_one_series(
                group,
                grain_id=str(material_id),
                prophet_params=PROPHET_PARAMS_MATERIAL,
            )
        except Exception as exc:
            print(f"  skip {material_id}: {exc}")
            continue

        _write_backtest(material_run_id, str(material_id), metrics, detail)
        n_rows = _write_forecast(
            material_run_id, as_of_date, "material", str(material_id), forecast, cutoff
        )
        material_results.append({
            "material_id": material_id,
            "mape": metrics["mape"],
            "wmape": metrics["wmape"],
            "forecast_rows": n_rows,
        })
        print(
            f"  {material_id}: MAPE={metrics['mape']:.1f}%  "
            f"wMAPE={metrics['wmape']:.1f}%  rows={n_rows}"
        )

    if material_results:
        summary = pd.DataFrame(material_results)
        mlflow.log_metric("test_wmape_median", float(summary["wmape"].median()))
        mlflow.log_metric("skus_trained", len(summary))
    mlflow.set_tag("run_id", str(material_run_id))

print(f"Material run_id={material_run_id}")
if material_results:
    display(pd.DataFrame(material_results).sort_values("wmape"))
else:
    print("No material models trained")

## Review outputs

In [ ]:
display(spark.sql(f"""
    SELECT run_id, run_name, as_of_date, grain, target_column, created_at
    FROM {ML}.forecast_runs
    ORDER BY run_id DESC
    LIMIT 5
"""))

display(spark.sql(f"""
    SELECT year_month,
           ROUND(yhat) AS forecast,
           ROUND(yhat_lower) AS lower_bound,
           ROUND(yhat_upper) AS upper_bound
    FROM {ML}.sales_forecast_monthly
    WHERE run_id = {company_run_id}
    ORDER BY forecast_month
"""))

## Company forecast chart

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

hist = company_df[["month_start_date", TARGET_COLUMN]].rename(
    columns={"month_start_date": "ds", TARGET_COLUMN: "y"}
)
fc = company_forecast[company_forecast["ds"] > company_cutoff]
cutoff = company_cutoff

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(hist["ds"], hist["y"], label="Historical", color="steelblue", marker="o", ms=4, linewidth=1.5)
ax.plot(fc["ds"], fc["yhat"], label="Forecast", color="tomato", linewidth=2)
ax.fill_between(fc["ds"], fc["yhat_lower"], fc["yhat_upper"], alpha=0.2, color="tomato", label="95% CI")
ax.axvline(cutoff, linestyle="--", color="gray", linewidth=1.2, label="Forecast Start")
ax.set_title("Company Sales Quantity — Historical & 12-Month Forecast", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Sales Quantity")
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate(rotation=30)
plt.tight_layout()
display(fig)
plt.close(fig)